# Caso B · 03 Features para forecasting horario

> _Tutorial · Caso de uso: **B — Forecast consumo 24h** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir el dataset de features (oro) para entrenar modelos de predicción del consumo eléctrico a 24h. Generar lags, rolling, calendario lectivo y variables exógenas (T_ext, GHI).


## 2. Qué se aprende

- Lag features (1h, 24h, 168h).
- Rolling means (7d, 24h).
- Codificación cíclica de hora/día.
- Variables exógenas y posibles fugas de información.


## 3. Contexto del caso de uso

Los modelos SARIMA / XGBoost / LSTM esperan formatos distintos. Aquí construimos un **dataset largo** con todas las features y un esquema tabular en cuatro columnas (X, y, time, asset).


## 4. Relación con CENTINELA+

Las mismas features se calculan en producción cada hora antes de invocar el modelo. La función `make_features(df)` debe ser pura para reproducibilidad.


## 5. Relación con Medallion

**Capa oro** específica del Caso B.


## 6. Datos de entrada

Mock `bdg2_education_subset_mock.csv`. En modo online, se leería de Influx.


## 7. Schema CAPTIA esperado

Las features no se publican como `captia_point`; viven en pandas / Parquet.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos y pivotamos a un DataFrame ancho.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "bdg2_education_subset_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df = df[df.building_id == df.building_id.unique()[0]].sort_values("timestamp").set_index("timestamp")
df.head()


## 10. Exploración paso a paso

Verificamos cobertura horaria sin huecos.


In [ ]:
gaps = df.index.to_series().diff().dt.total_seconds().dropna()
print("Mediana entre puntos (s):", gaps.median(), "  Máximo:", gaps.max())


## 11. Transformación bronce → plata

Construimos features (no escribimos a InfluxDB; el oro vive como Parquet).


In [ ]:
def make_features(d):
    out = pd.DataFrame(index=d.index)
    out["y"] = d["power_kw"]
    out["t_outdoor"] = d["t_outdoor"]
    out["ghi"] = d["ghi"]
    # Cyclical
    h = d.index.hour
    out["hour_sin"] = np.sin(2 * np.pi * h / 24)
    out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["dow"] = d.index.dayofweek
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    # Lags
    out["lag_1h"] = d["power_kw"].shift(1)
    out["lag_24h"] = d["power_kw"].shift(24)
    out["lag_168h"] = d["power_kw"].shift(168)
    # Rolling
    out["roll_24h_mean"] = d["power_kw"].shift(1).rolling(24).mean()
    out["roll_168h_mean"] = d["power_kw"].shift(1).rolling(168).mean()
    return out.dropna()

X = make_features(df)
X.head()


## 12. Construcción de capa oro

Persistimos a Parquet (oro local).


In [ ]:
out_dir = ROOT / "output" / "case_B"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "features_b0.parquet"
X.to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)} ({len(X)} filas)")


## 13. Visualizaciones explicativas

Mostramos las correlaciones más altas con `y`.


In [ ]:
correls = X.drop(columns=["y"]).apply(lambda c: c.corr(X["y"])).sort_values()
correls.plot.barh(figsize=(7, 4), color="#FF5722")
plt.title("Correlación de features con y (power_kw)")
plt.tight_layout()


## 14. Validaciones

Sin NaN tras `dropna`, lags correctos.


In [ ]:
assert X.isna().sum().sum() == 0
assert X["lag_24h"].iloc[0] == X["y"].iloc[0] - (X["y"].iloc[0] - X["lag_24h"].iloc[0])  # tautología, pero comprueba shape
print("Features shape:", X.shape)


## 15. Errores comunes

1. **Leakage temporal**: usar `.rolling()` sin `shift(1)` mezcla pasado y futuro.
2. **Codificación de `dow` no cíclica**: lunes y domingo aparecerán muy lejos en el espacio de features. Usar sen/cos.
3. **Imputar NaN con la media**: para series temporales mejor `ffill` o drop.


## 16. Ejercicios propuestos

1. Añade `lag_24h_diff = y - lag_24h` y mide su correlación.
2. Crea una feature de calendario lectivo (Comunidad Valenciana).
3. Compara MAE de un modelo con y sin features cíclicas.


## 17. Cómo se reutiliza con datos reales

`make_features(df)` es pura: misma firma, distinto origen. La única adaptación es la columna `power_kw` ↔ `power_01` en CENTINELA+.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `02_case_B_energy_forecasting/04_baseline_sarima_xgboost_lstm.ipynb`.
- Documento web del caso: `docs/use-cases/case-b-energy-forecasting.md`.
